# Logistic Regression from Scratch
**Zahraa Al-Naami | Break Through Tech AI coursework portfolio**

Reconstructed from my saved Lab 4 HTML export because the original course workspace is inaccessible. Course-provided scaffolding and the original exercise informed this implementation; this is coursework, not an independently designed production system. AI assistance was used for this portfolio reconstruction and documentation.

**Provenance:** saved outputs below are historical outputs recovered from the HTML, not newly executed results. The exported feature-selection cell was malformed; the missing `feature_list` assignment name has been restored. Other model code is preserved. Blank course analysis/reflection prompts were replaced by clearly identified portfolio commentary; no personal reflection or past AI-use claim was invented.

## Goal and data
Implement binary logistic regression for Airbnb superhost status and compare coefficients and fitting time with scikit-learn. The seven input features are review scores and host response/acceptance rates. The course supplied preprocessed data (imputed, encoded, and scaled). Place the authorized course CSV at `data/airbnbData_train.csv` before running. The CSV is not included and no public download source has been verified.

## Method
The implementation uses a sigmoid probability function, gradient, Hessian, and **Newton's method** weight updates. Although the exercise described gradient descent, an inverse-Hessian update is a Newton step. All examples are used for fitting; there is no held-out evaluation in this lab.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import os 
from sklearn.linear_model import LogisticRegression


## Model implementation
Preserved from the export. This educational implementation explicitly inverts the Hessian and uses a normalized-weight stopping criterion; see limitations below.

In [ ]:
class LogisticRegressionScratch(object):
    
    def __init__(self, tolerance = 10**-8, max_iterations = 20):
        
        self.tolerance = tolerance
        self.max_iterations = max_iterations
        self.weights_array = None # holds current weights and intercept (intercept is at the last position)
        self.prior_w = None # holds previous weights and intercept (intercept is at the last position)
        
        # once we are done training, these variables will hold the 
        # final values for the weights and intercept
        self.weights = None
        self.intercept = None 

        
    def predict_proba(self, X):
        '''
        Compute probabilities using the inverse logit
        - Inputs: The Nx(K+1) matrix with intercept column X
        - Outputs: Vector of probabilities of length N
        '''
    
        
        XW = X.dot(self.weights_array)
        P = 1/(1 + np.exp(-XW))

        
        return P

    
    
    def compute_gradient(self, X, Y, P):
        '''
        Computes the gradient vector
        -Inputs:
            - The Nx(K+1) matrix with intercept column X
            - Nx1 vector y (label) 
            - Nx1 vector of predictions P
        -Outputs: 1x(K+1) vector of gradients
        '''
        
        
        ### STEP B - WRITE YOUR CODE HERE
        G = -(Y - P).dot(X)
                
        return G
        
    def compute_hessian(self, X, P):
        '''
        computes the Hessian matrix
        -inputs:
            - Nx(K+1) matrix X
            - Nx1 vector of predictions P
        -outputs:
            - KxK Hessian matrix H=X^T * Diag(Q) * X
        '''

        ### STEP C - WRITE YOUR CODE HERE
        Q = P * (1-P)
        XQ = X.T*Q
        H = XQ.dot(X)
        
        return H


    def update_weights(self, X, y):
        '''
        Updates existing weight vector
        -Inputs:
            -Nx(Kx1) matrix X
            -Nx1 vector y
        -Calls predict_proba, compute_gradient and compute_hessian and uses the 
        return values to update the weights array
        '''
        
        
        ### STEP D - WRITE YOUR CODE HERE
    
        P = self.predict_proba(X)
        G = self.compute_gradient(X, y, P)
        H = self.compute_hessian(X, P)
        self.prior_w = self.weights_array.copy()
        self.weights_array = self.weights_array -  np.linalg.inv(H).dot(G)
      
           
    def check_stop(self):
        '''
        check to see if euclidean distance between old and new weights (normalized)
        is less than the tolerance
        
        returns: True or False on whether stopping criteria is met
        '''
        
        ### STEP E - WRITE YOUR CODE HERE
        w_old_norm = self.prior_w / np.linalg.norm(self.prior_w)
        w_new_norm = self.weights_array / np.linalg.norm(self.weights_array)
        diff = w_old_norm - w_new_norm
        distance = np.sqrt(diff.dot(diff))
        stop = distance < self.tolerance
        
        return stop
        
        
    def fit(self, X, y):
        '''
        X is the Nx(K-1) data matrix
        Y is the labels, using {0,1} coding
        '''
        
        #set initial weights - add an extra dimension for the intercept
        self.weights_array = np.zeros(X.shape[1] + 1)
        
        #Initialize the slope parameter to log(base rate/(1-base rate))
        self.weights_array[-1] = np.log(y.mean() / (1-y.mean()))
        
        #create a new X matrix that includes a column of ones for the intercept
        X_int = np.hstack((X, np.ones((X.shape[0],1))))

        for i in range(self.max_iterations):
            self.update_weights(X_int, y)
            
            # check whether we should
            stop = self.check_stop()
            if stop:
                # since we are stopping, lets save the final weights and intercept
                self.set_final_weights()
                self.set_final_intercept()
                break
                
    
    def set_final_weights(self):
        self.weights = self.weights_array[0:-1]
        
    def set_final_intercept(self):
        self.intercept = self.weights_array[-1]  
        
    def get_weights(self):
        return self.weights
    
    def get_intercept(self):
        return self.intercept
        


## Load the course data and select features

In [ ]:
filename = os.path.join(os.getcwd(), "data", "airbnbData_train.csv")


In [ ]:
df = pd.read_csv(filename)


In [ ]:
feature_list = ['review_scores_rating','review_scores_cleanliness','review_scores_checkin','review_scores_communication','review_scores_value','host_response_rate','host_acceptance_rate']
feature_list


['review_scores_rating',
 'review_scores_cleanliness',
 'review_scores_checkin',
 'review_scores_communication',
 'review_scores_value',
 'host_response_rate',
 'host_acceptance_rate']

In [ ]:
X = df[feature_list]

y = df['host_is_superhost']


## Fit the scratch model

In [ ]:
lr = LogisticRegressionScratch()
lr.fit(X, y)


In [ ]:
print('The fitted weights and intercept are:')
print(lr.get_weights(), lr.get_intercept())


The fitted weights and intercept are:
[ 0.56690005  0.492255    0.201587    0.25551467 -0.00590516  1.71592957
  0.26478817] -1.829062262272181


## Compare coefficients with scikit-learn
`C=10**10` approximates very weak regularization; coefficient agreement is approximate, not a proof of correctness.

In [ ]:
lr_sk = LogisticRegression(C=10**10)
lr_sk.fit(X, y)


LogisticRegression(C=10000000000, class_weight=None, dual=False,
                   fit_intercept=True, intercept_scaling=1, l1_ratio=None,
                   max_iter=100, multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=None, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)

In [ ]:
print('The fitted weights and intercept with sklearn are:')
print(lr_sk.coef_, lr_sk.intercept_)


The fitted weights and intercept with sklearn are:
[[ 0.56691547  0.49224905  0.20150113  0.25563246 -0.005929    1.71592022
   0.26479199]] [-1.82906726]


## Recorded runtime benchmark
IPython `%timeit` commands require Jupyter/Colab. These historical measurements are environment-dependent and have substantial variability.

In [ ]:
%timeit lr.fit(X, y)


The slowest run took 4.16 times longer than the fastest. This could mean that an intermediate result is being cached.
296 ms ± 153 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%timeit lr_sk.fit(X, y)


333 ms ± 78 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Portfolio interpretation (added during reconstruction)
The scratch and scikit-learn coefficient vectors are close. For example, their first coefficients are approximately 0.566900 and 0.566915, and their intercepts are approximately -1.829062 and -1.829067. This supports agreement for this dataset and configuration, but does not establish correctness for all inputs.

The recorded fitting times were 296 ± 153 ms for the scratch implementation and 333 ± 78 ms for scikit-learn (7 runs, 1 loop each). Their variability overlaps, so these measurements do not justify a reliable speed advantage.

## Limitations and next steps
- No held-out test accuracy, F1, or generalization claim is available.
- The original data and dependency versions have not been recovered; the notebook has been checked for Python syntax but has not been rerun on the original data.
- Explicit Hessian inversion can fail with singular or poorly conditioned matrices. A linear solve with suitable stabilization would be a future improvement.
- Normalizing a zero weight vector can cause division by zero. Convergence checks need strengthening.
- The preserved `fit` method only assigns final coefficient attributes when its convergence criterion triggers; reaching the iteration limit needs explicit handling.
- Future validation should include numerical gradient checks, edge cases, convergence diagnostics, and a train/validation/test workflow with preprocessing fitted only on training data.

## Attribution
Adapted from Break Through Tech AI Lab 4, “ML Life Cycle: Modeling — Building a Logistic Regression Model From Scratch.” The course supplied the exercise, scaffolding, and preprocessed data. This reconstructed portfolio records the submitted implementation and saved outputs and documents its limitations. It is not production-ready.